# NDT7 (M-Lab) Data Prep — Philippines Broadband + Mobile, Region x Quarter

**Not yet executed in this repo** — Philippines raw data (`data/ph/mlab_ph_clean.parquet`, ~262M raw NDT7
test records) is not available locally. This notebook restructures/ports the already-verified
aggregation logic from `notebooks/ndt7/ph/ndt7_ph_main.ipynb` and `ndt7_ph_paper.ipynb` (both
executed with real outputs by whoever has the full local dataset) into the tigger prep/EDA split
used by Cambodia/Thailand/Vietnam. Needs a run+verify pass before use.

**Two structural differences from Cambodia/Thailand/Vietnam, both deliberate, not bugs:**

1. **No zoom-16 tile-binning.** The PH source notebooks never built a tile-binning step — the
   v2 parquet already carries clean `province`/`region` columns from a GADM point-in-polygon
   join, and every existing PH analysis aggregates directly from those columns. Adding
   Ookla-style tile-binning here would be new, unverified geo code with no way to check it
   locally, so `n_tiles` is left as `NaN` in the export and **`is_reliable` uses `total_tests
   >= 100` only** (not `total_tests>=100 & n_tiles>=5` like KH/TH/VN) — flagged explicitly so
   downstream notebooks don't silently assume tile-based reliability.
2. **Output granularity is region (17), not raw ADM1 province (80).** `data/reference/philippines_reference.csv`
   and `data/geo/philippines_provinces.geojson` only exist at the 17-region level — this matches
   what `notebooks/ookla/philippines_eda.ipynb` already does for Ookla PH (its `province` column
   is literally the 17 region names, e.g. `ARMM`, `Bicol Region`). The raw NDT7 parquet's `region`
   column uses GADM administrative codes (`Region I`, `BARMM`, ...) instead of the descriptive
   names in the reference CSV/geojson, so a name-mapping step (analogous to Cambodia's
   `PROVINCE_MAP`) is applied below. True 80-province-level detail is NOT lost — it lives in the
   PH-specific sections appended at the end of `philippines_eda.ipynb` (Manila vs rest, top/bottom
   provinces, island-group divide), which is exactly why the user asked to keep those as
   PH-specific additions rather than force a 1:1 template match.

**Outputs:**
- `data/exports/ndt7_philippines_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_philippines_province_quarterly.csv` — Mobile/Cellular


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ph/mlab_ph_clean.parquet'
PH_REF_CSV = '../../../data/reference/philippines_reference.csv'


### 1. Region Code Mapping — Raw (GADM code) → Reference (descriptive name)

Raw parquet `region` values (`NCR`, `CAR`, `Region I`...`Region XIII`, `MIMAROPA`, `BARMM`) are
standard PH Regional Development Council codes; the reference CSV/geojson use the equivalent
descriptive names (`Ilocos Region`, `Cagayan Valley`, ...). This mapping is the standard,
publicly documented PH region naming — the same 1:1 correspondence used throughout Philippine
government statistics, not something invented for this pipeline.


In [ ]:
REGION_MAP = {
    'NCR': 'NCR',
    'CAR': 'CAR',
    'Region I': 'Ilocos Region',
    'Region II': 'Cagayan Valley',
    'Region III': 'Central Luzon',
    'Region IV-A': 'Calabarzon',
    'MIMAROPA': 'Mimaropa',
    'Region V': 'Bicol Region',
    'Region VI': 'Western Visayas',
    'Region VII': 'Central Visayas',
    'Region VIII': 'Eastern Visayas',
    'Region IX': 'Zamboanga Peninsula',
    'Region X': 'Northern Mindanao',
    'Region XI': 'Davao Region',
    'Region XII': 'Soccsksargen',
    'Region XIII': 'Caraga',
    'BARMM': 'ARMM',
}

ref_check = pd.read_csv(PH_REF_CSV)
unmapped_targets = set(REGION_MAP.values()) - set(ref_check['province_en'])
print(f"Mapping covers {len(REGION_MAP)} raw region codes -> {len(set(REGION_MAP.values()))} reference regions")
if unmapped_targets:
    print(f"WARNING — mapped targets not found in reference: {unmapped_targets}")
else:
    print("All mapped targets found in reference.")


### 2. Region x Quarter Aggregation (DuckDB)

Reuses the exact per-quarter aggregation shape already verified in `ndt7_ph_main.ipynb`
(`region_speed_figure`) and `ndt7_ph_paper.ipynb` (Part 1 trend query) — average-per-quarter,
grouped by region/network_type/direction — extended to also emit **upload** (same query shape,
just the other already-present `type` value; the source notebooks only ever looked at download,
so this exact query hasn't been run before, but it is structurally identical to code that has).
One monitoring/bot client IP (`58.69.220.245`, 865k tests, all in SultanKudarat — documented in
`ndt7_ph_manila.ipynb` Part 3) is excluded, same as every PH source notebook.


In [ ]:
FLOOD_IPS = ('58.69.220.245',)
_flood = ", ".join(f"'{ip}'" for ip in FLOOD_IPS)

sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps, LEAST(min_rtt, 2000) AS min_rtt,
        type, network_type, province, region,
        year, CAST(CEIL(month / 3.0) AS INT) AS qtr
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND province IS NOT NULL AND region IS NOT NULL
      AND network_type IN ('broadband', 'cellular')
      AND client_ip NOT IN ({_flood})
)
SELECT
    region, network_type, type,
    (CAST(year AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
    AVG(mean_throughput_mbps) AS avg_thr,
    AVG(CASE WHEN min_rtt < 2000 THEN min_rtt END) AS avg_lat,
    COUNT(*) AS test_count
FROM filtered
GROUP BY region, network_type, type, year_q
"""

con = duckdb.connect()
region_q_all = con.execute(sql).df()
region_q_all['region'] = region_q_all['region'].map(REGION_MAP)
region_q_all = region_q_all.dropna(subset=['region'])
print(f"Region x quarter x type x network rows: {len(region_q_all):,}")
print(f"Quarters covered: {sorted(region_q_all['year_q'].unique())}")
print(region_q_all['network_type'].value_counts())


### 3. Region-Level Weighted Aggregation (per network type) + Reference Merge

Pivots download/upload into wide columns, merges reference GDP/density/tier data, and computes
`is_reliable`. **No `n_tiles` for Philippines** (see notebook intro) — `is_reliable` here is
`total_tests >= 100` only.


In [ ]:
def build_region_quarterly(region_q_all, network_type, ref):
    d = region_q_all[region_q_all['network_type'] == network_type]
    print(f"[{network_type}] region x quarter x type rows: {len(d):,}")

    dl = d[d['type'] == 'download'].rename(columns={'avg_thr': 'avg_d_mbps', 'avg_lat': 'avg_lat_ms_wt', 'test_count': 'total_tests'})
    ul = d[d['type'] == 'upload'].rename(columns={'avg_thr': 'avg_u_mbps'})

    dl_stats = dl[['year_q', 'region', 'avg_d_mbps', 'avg_lat_ms_wt', 'total_tests']]
    ul_stats = ul[['year_q', 'region', 'avg_u_mbps']]

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'region'], how='outer')
    master = master.rename(columns={'year_q': 'quarter', 'region': 'province'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)
    master['n_tiles'] = np.nan   # not applicable for Philippines — see notebook intro

    master['is_reliable'] = master['total_tests'] >= 100
    print(f"[{network_type}] region x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — regions with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

ref = pd.read_csv(PH_REF_CSV)


---
## Part 1 — Broadband


In [ ]:
broadband_master = build_region_quarterly(region_q_all, 'broadband', ref)
broadband_master.head()


In [ ]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_philippines_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)


---
## Part 2 — Mobile/Cellular


In [ ]:
mobile_master = build_region_quarterly(region_q_all, 'cellular', ref)
mobile_master.head()


In [ ]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_philippines_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)


## Summary

- Input: Philippines NDT7 raw test records (`data/ph/mlab_ph_clean.parquet`), already
  province/region-joined + ISP-classified — **not present locally, needs teammate run+verify**.
- Output: region x quarter aggregates (17 PH regions, matching Ookla PH's own convention and the
  only granularity with GDP/tier reference data) for Broadband and Mobile separately.
- **Reliability differs from KH/TH/VN**: `is_reliable = total_tests>=100` only — no tile-binning,
  no `n_tiles` (see notebook intro for why). Not directly comparable to the other tigger
  countries' `n_tiles>=5` bar; treat as its own methodology, not a relaxed version of theirs.
- True ADM1-province-level (80 provinces) and city-level detail is NOT covered by this
  export — it lives in the PH-specific sections of `philippines_eda.ipynb`.
